In [ ]:
import json
import time
import os
from groq import Groq
from datasets import load_dataset

# 1. Initialize the Groq Client
# Paste your actual Groq API key string here, or set it in Colab's secret manager
GROQ_API_KEY = "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"
client = Groq(api_key=GROQ_API_KEY)

# Download the dataset split if not already loaded in the environment
print("Loading base C++ dataset...")
base_dataset = load_dataset("codeparrot/xlcost-text-to-code", "C++-snippet-level", split="train")

def get_teacher_model_analysis(cpp_code):
    """
    Sends the raw C++ code to Llama 3 on Groq, enforcing a strict JSON output schema.
    """
    prompt = f"""
    You are an expert C++ algorithmic analyzer. Read the following code:
    ```cpp
    {cpp_code}
    ```
    Output your analysis strictly in valid JSON format with exactly three keys:
    "docstring": A one-sentence professional summary of what the code does starting with an active verb.
    "time_complexity": The worst-case Big O time complexity (e.g., "O(N)", "O(N log N)").
    "space_complexity": The worst-case Big O space complexity (e.g., "O(1)", "O(N)").
    Do not output any markdown formatting, backticks, or extra text. Just the raw JSON object.
    """

    # Call the live Groq API
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1, # Low temperature minimizes hallucinations
        response_format={"type": "json_object"} # Enforces JSON mode at the API level
    )

    # Parse and return the JSON response string
    return json.loads(completion.choices[0].message.content)

# 2. Run the Processing Factory
target_samples = 2500
output_filename = "augmented_cpp_tasks.jsonl"

# Create file if it doesn't exist
if not os.path.exists(output_filename):
    open(output_filename, "w").close()

# Resume support
generated = 0
with open(output_filename, "r", encoding="utf-8") as f:
    generated = sum(1 for _ in f)

print(f"Already have {generated} samples.")
print(f"Beginning live synthetic generation for {target_samples} samples...")

for i in range(generated, min(target_samples, len(base_dataset))):
    sample = base_dataset[i]
    raw_code = sample["code"]

    # Skip overly long snippets
    if len(raw_code) > 800 or len(raw_code.strip()) == 0:
        continue

    try:
        analysis = get_teacher_model_analysis(raw_code)

        instruction = (
            "Analyze the following C++ code. Provide a professional docstring detailing its "
            "functionality, and determine its worst-case Time and Space complexity.\n\n"
            f"```cpp\n{raw_code.strip()}\n```"
        )

        output = (
            "<analysis>\n"
            f"<docstring>\n{analysis['docstring']}\n</docstring>\n"
            f"<time_complexity>{analysis['time_complexity']}</time_complexity>\n"
            f"<space_complexity>{analysis['space_complexity']}</space_complexity>\n"
            "</analysis>"
        )

        item = {
            "instruction": instruction,
            "output": output
        }

        # Write immediately to disk
        with open(output_filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

        generated += 1

        if generated % 10 == 0:
            print(f"Generated {generated} clean profiles successfully...")

        time.sleep(0.2)

    except Exception as e:
        print(f"Sample index {i} encountered an error: {e}")

        # Wait longer on rate limits
        if "429" in str(e):
            print("Rate limit reached. Sleeping for 60 seconds...")
            time.sleep(60)

        continue

print(
    f"\nPipeline execution complete! "
    f"Saved {generated} samples to '{output_filename}'."
)

In [ ]:
import json
import re

def verify_dataset(filename="augmented_cpp_tasks.jsonl"):
    total_records = 0
    malformed_json = 0
    missing_tags = 0
    valid_records = 0

    # Required components in our structured target schema
    required_tags = [
        r"<analysis>",
        r"</analysis>",
        r"<docstring>",
        r"</docstring>",
        r"<time_complexity>.*?</time_complexity>",
        r"<space_complexity>.*?</space_complexity>"
    ]

    print(f"=== Initiating Verification for {filename} ===")

    with open(filename, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, 1):
            total_records += 1
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                print(f"[Error] Line {line_idx}: Invalid JSON structure.")
                malformed_json += 1
                continue

            # Verify top-level keys
            if "instruction" not in data or "output" not in data:
                print(f"[Error] Line {line_idx}: Missing 'instruction' or 'output' key.")
                missing_tags += 1
                continue

            # Validate output tag structure
            output_text = data["output"]
            is_valid_structure = True
            for pattern in required_tags:
                if not re.search(pattern, output_text, re.DOTALL):
                    is_valid_structure = False
                    break

            if not is_valid_structure:
                print(f"[Warning] Line {line_idx}: Output fails XML schema validation.")
                missing_tags += 1
                continue

            valid_records += 1

    # Print Final Telemetry Report
    print("\n=== DATASET HEALTH REPORT ===")
    print(f"Total Rows Processed:      {total_records}")
    print(f"Successfully Validated:    {valid_records} ({ (valid_records/total_records)*100:.1f}%)")
    print(f"Malformed JSON Errors:     {malformed_json}")
    print(f"Schema/Tag Violations:     {missing_tags}")

    if valid_records > 0:
        print("\n=== PRINTING FIRST VALID SAMPLE FOR VISUAL INSPECTION ===")
        with open(filename, "r", encoding="utf-8") as f:
            first_sample = json.loads(f.readline())
            print("\n>> PROMPT TARGET INSTRUCTION:")
            print(first_sample["instruction"][:300] + "\n... [Code Truncated for Preview] ...")
            print("\n>> TARGET OUTPUT FORMAT:")
            print(first_sample["output"])

# Execute the validation check
verify_dataset("augmented_cpp_tasks.jsonl")

In [1]:
# FIX: Install compatible versions and restart session if error persists
!pip install -U "huggingface_hub>=0.25.0" "transformers>=4.45.0" "datasets" "peft" "bitsandbytes" "accelerate"

import json
import re
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Initialize Storage & Paths
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted.")
checkpoint_dir = "/content/drive/MyDrive/AlgorithmicCodeAnalyst_Checkpoints"

# 2. Setup Base Model & Tokenizer
model_id = "bigcode/starcoder2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# 3. Dynamic Data Sanitation & Tokenization
dataset = load_dataset("json", data_files="augmented_cpp_tasks.jsonl", split="train").shuffle(seed=42)

def clean_and_tokenize(examples):
    text_batches = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        inst = inst.replace("NEW_LINE", "\n")
        out = out.replace("Analyzing the given function to determine", "Calculates")
        full_text = f"{inst}\n\n{out}{tokenizer.eos_token}"
        text_batches.append(full_text)

    tokenized = tokenizer(text_batches, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(clean_and_tokenize, batched=True, remove_columns=["instruction", "output"])

# 4. Inject LoRA Adapters
peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.config.bos_token_id = tokenizer.eos_token_id
model.generation_config.bos_token_id = tokenizer.eos_token_id
# 5. Training Configurations
training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    fp16=True,
    num_train_epochs=3,
    lr_scheduler_type="constant",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

Mounted at /content/drive


config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

...

Map:   0%|          | 0/2199 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.431906
10,2.682596
15,1.590911
20,1.314199
25,1.014249
30,0.729456
35,0.703599
40,0.711539
45,0.603180
50,0.652389


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=825, training_loss=0.4024626727537675, metrics={'train_runtime': 5956.8798, 'train_samples_per_second': 1.107, 'train_steps_per_second': 0.138, 'total_flos': 5.853767675255194e+16, 'train_loss': 0.4024626727537675, 'epoch': 3.0})

In [2]:
!nvidia-smi

Sun Jun 28 16:02:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             42W /   70W |    3953MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Define your final permanent storage paths
local_save_dir = "./final_algorithmic_analyst_model"
drive_save_dir = "/content/drive/MyDrive/AlgorithmicCodeAnalyst_Final"

# 1. Save locally to the Colab workspace container
print("Saving model adapters and tokenizer locally...")
trainer.model.save_pretrained(local_save_dir)
tokenizer.save_pretrained(local_save_dir)

# 2. Save a secure backup copy directly to Google Drive
print("Saving a secure backup copy to Google Drive...")
trainer.model.save_pretrained(drive_save_dir)
tokenizer.save_pretrained(drive_save_dir)

print("🎉 Model successfully saved and backed up!")

Saving model adapters and tokenizer locally...
Saving a secure backup copy to Google Drive...
🎉 Model successfully saved and backed up!


In [4]:
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Mount Storage & Define Paths
drive.mount('/content/drive')
base_model_id = "bigcode/starcoder2-3b"
adapter_dir = "/content/drive/MyDrive/AlgorithmicCodeAnalyst_Final"

print("Initializing tokenizer and 4-bit base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Match the exact quantization configuration used during training
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 2. Inject Your Fine-Tuned LoRA Adapters
print("Injecting custom fine-tuned LoRA weights from Google Drive...")
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval() # Set model to evaluation mode

# 3. Define an Unseen C++ Test Algorithm
# Let's test it with a standard dynamic programming / array problem
test_cpp_code = """
int findMaxSubArraySum(vector<int>& nums) {
    int max_so_far = nums[0];
    int curr_max = nums[0];
    for (size_t i = 1; i < nums.size(); i++) {
        curr_max = max(nums[i], curr_max + nums[i]);
        max_so_far = max(max_so_far, curr_max);
    }
    return max_so_far;
}
"""

# 4. Format Prompt to Match Training Schema Exactly
prompt = (
    "Analyze the following C++ code. Provide a professional docstring detailing its "
    "functionality, and determine its worst-case Time and Space complexity.\n\n"
    f"```cpp\n{test_cpp_code.strip()}\n```\n\n<analysis>\n"
)

# 5. Run Text Generation Inference
print("\nRunning inference engine...")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,         # Low temperature enforces structure
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and extract the output format
full_generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Cleanly print out just the target structured answer block
print("\n" + "="*40)
print("🤖 MODEL PREDICTION OUTPUT:")
print("="*40)
# Extract starting from our open analysis tag to filter out the prompt copy
if "<analysis>" in full_generated_text:
    print("<analysis>" + full_generated_text.split("<analysis>")[-1])
else:
    print(full_generated_text)
print("="*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initializing tokenizer and 4-bit base model...


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Injecting custom fine-tuned LoRA weights from Google Drive...


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Running inference engine...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🤖 MODEL PREDICTION OUTPUT:
<analysis>
<docstring>
Finds the maximum sum of a subarray in the given array.
</docstring>
<time_complexity>O(N)</time_complexity>
<space_complexity>O(1)</space_complexity>
</analysis>

<docstring>
Calculates the maximum sum of a subarray by iterating through the array and keeping track of the maximum sum seen so far.
</docstring>
<time_complexity>O(N)</time_complexity>
<space_complexity>O(1)</space_complexity>
</analysis>
/cpp/find_max_subarray_sum.cpp
#include <bits/stdc++.h


In [5]:
full_generated_text

'Analyze the following C++ code. Provide a professional docstring detailing its functionality, and determine its worst-case Time and Space complexity.\n\n```cpp\nint findMaxSubArraySum(vector<int>& nums) {\n    int max_so_far = nums[0];\n    int curr_max = nums[0];\n    for (size_t i = 1; i < nums.size(); i++) {\n        curr_max = max(nums[i], curr_max + nums[i]);\n        max_so_far = max(max_so_far, curr_max);\n    }\n    return max_so_far;\n}\n```\n\n<analysis>\n<docstring>\nFinds the maximum sum of a subarray in the given array.\n</docstring>\n<time_complexity>O(N)</time_complexity>\n<space_complexity>O(1)</space_complexity>\n</analysis>\n\n<docstring>\nCalculates the maximum sum of a subarray by iterating through the array and keeping track of the maximum sum seen so far.\n</docstring>\n<time_complexity>O(N)</time_complexity>\n<space_complexity>O(1)</space_complexity>\n</analysis>\n/cpp/find_max_subarray_sum.cpp\n#include <bits/stdc++.h'

In [6]:
# Robust parsing to trap the exact XML block and destroy the ramblings
if "</analysis>" in full_generated_text:
    # 1. Split to get everything after our prompt's opening tag
    after_open = full_generated_text.split("<analysis>")[-1]
    # 2. Split again to chop off the rambling after the closing tag
    clean_core = after_open.split("</analysis>")[0]

    # 3. Print the perfectly reconstructed block
    print("<analysis>" + clean_core.strip() + "\n</analysis>")
else:
    print(full_generated_text)

print("="*40)

<analysis><docstring>
Finds the maximum sum of a subarray in the given array.
</docstring>
<time_complexity>O(N)</time_complexity>
<space_complexity>O(1)</space_complexity>
</analysis>


In [11]:
test_cpp_code = """
 int dude(vector<int>& arr) {
        sort(arr.begin(), arr.end());
        int ans = 1;

        for (int i = 1; i < arr.size(); i++) {
            if (arr[i] >= ans + 1) {
                ans++;
            }
        }

        return ans;
    }
"""

# 4. Format Prompt to Match Training Schema Exactly
prompt = (
    "Analyze the following C++ code. Provide a professional docstring detailing its "
    "functionality, and determine its worst-case Time and Space complexity.\n\n"
    f"```cpp\n{test_cpp_code.strip()}\n```\n\n<analysis>\n"
)

# 5. Run Text Generation Inference
print("\nRunning inference engine...")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,         # Low temperature enforces structure
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and extract the output format
full_generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


Running inference engine...


In [12]:
# Robust parsing to trap the exact XML block and destroy the ramblings
if "</analysis>" in full_generated_text:
    # 1. Split to get everything after our prompt's opening tag
    after_open = full_generated_text.split("<analysis>")[-1]
    # 2. Split again to chop off the rambling after the closing tag
    clean_core = after_open.split("</analysis>")[0]

    # 3. Print the perfectly reconstructed block
    print("<analysis>" + clean_core.strip() + "\n</analysis>")
else:
    print(full_generated_text)

print("="*40)

<analysis><docstring>
Determines the maximum number of consecutive integers in an array that are sorted in ascending order.
</docstring>
<time_complexity>O(N)</time_complexity>
<space_complexity>O(1)</space_complexity>
</analysis>


In [13]:
!pip install gradio

In [ ]:
import gradio as gr
import torch

def analyze_code(cpp_code):
    """
    The function that Gradio will call whenever a user clicks 'Submit' on the Web UI.
    """
    # 1. Format the prompt exactly as the model expects
    prompt = (
        "Analyze the following C++ code. Provide a professional docstring detailing its "
        "functionality, and determine its worst-case Time and Space complexity.\n\n"
        f"```cpp\n{cpp_code.strip()}\n```\n\n<analysis>\n"
    )

    # 2. Tokenize and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 3. Generate the prediction
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 4. Robust Parsing (Fixing the over-generation issue)
    if "</analysis>" in full_text:
        after_open = full_text.split("<analysis>")[-1]
        clean_core = after_open.split("</analysis>")[0].strip()
        return f"<analysis>\n{clean_core}\n</analysis>"
    else:
        # Fallback if the model misses the tag completely
        return full_text

# 5. Build the Web Interface
with gr.Blocks() as demo:
    gr.Markdown("# 🚀 Algorithmic Code Analyst")
    gr.Markdown("Paste your C++ function below. The PEFT fine-tuned StarCoder2 model will analyze the logic, write a professional docstring, and predict the Big O Time and Space complexities.")

    with gr.Row():
        with gr.Column():
            input_code = gr.Code(language="cpp", label="Input C++ Code", lines=15)
            analyze_btn = gr.Button("Analyze Algorithm", variant="primary")

        with gr.Column():
            output_xml = gr.Code(language="html", label="Model Analysis (XML Output)", lines=15)

    # Link the button to the function
    analyze_btn.click(fn=analyze_code, inputs=input_code, outputs=output_xml)

# 6. Launch the Server and create a Public URL
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://db9086b1c6c43a140f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
pip install -U gradio
gradio deploy